# RetryWise: Payment Retry Simulation

This notebook compares three ways to handle failed payment requests: stop after one attempt, retry with idempotency, and retry without idempotency. The aim is to measure how retries affect confirmed payments, duplicate charges, request volume, and completion time.

The main failure case is a timeout after a payment has already been charged. The client has no confirmation, but sending another request can create a second charge unless the service recognizes the original operation.

All payments and service responses are simulated. No payment provider is contacted and no money moves.

## 1. Environment

Run the cells in order using a Python 3.12 kernel with pandas and matplotlib installed. The simulation uses a fixed random seed so results can be reproduced.

The local environment was checked with NumPy 1.26.4 and pandas 2.2.2. Package installation is kept outside the execution flow so running the notebook does not change the environment.

In [ ]:
from dataclasses import dataclass
from collections import Counter
from typing import Dict, List, Tuple
import random
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2027
random.seed(SEED)
print("Random seed:", SEED)

## 2. Payment scenarios

Generate 600 payments with amounts between $5 and $200. Each payment has a predefined sequence of responses, reused across policies so they face the same scenarios.

| Response | Meaning | Retry? |
|---|---|---|
| `ok` | Payment charged and response received | No |
| `rate_limit` | Service rejects the request temporarily | Yes |
| `server_error` | Temporary service failure | Yes |
| `timeout_before_commit` | Request times out before a charge | Yes |
| `timeout_after_commit` | Charge succeeds, but the response is lost | Yes, with the same idempotency key |
| `card_declined` | Payment declined | No |

The response weights are chosen for this experiment; they are not measured payment failure rates. Amounts are stored as integer cents and converted to dollars for display.

In [ ]:
OUTCOMES = ("ok", "rate_limit", "server_error", "timeout_before_commit", "timeout_after_commit", "card_declined")
TRANSIENT = {"rate_limit", "server_error", "timeout_before_commit", "timeout_after_commit"}

@dataclass(frozen=True)
class Payment:
    payment_id: str
    amount_cents: int
    script: Tuple[str, ...]


def make_sample_payments(n=600, seed=SEED):
    rng = random.Random(seed)
    payments = []
    for i in range(n):
        amount = rng.randint(500, 20000)
        initial = rng.choices(
            OUTCOMES,
            weights=[70, 7, 7, 7, 4, 5],
            k=1,
        )[0]
        # Script outcomes for up to five attempts; later attempts are usually successful.
        followups = tuple(rng.choices(OUTCOMES, weights=[84, 3, 4, 3, 2, 4], k=4))
        payments.append(Payment(f"pay_{i:04d}", amount, (initial,) + followups))
    return payments

payments = make_sample_payments()
sample = pd.DataFrame([{"payment_id": p.payment_id, "amount_usd": p.amount_cents / 100,
                        "first_outcome": p.script[0]} for p in payments])
display(sample.head(8))
print("First-attempt outcome counts:", sample.first_outcome.value_counts().to_dict())

## 3. Simulated payment service

The service tracks requests separately from charges. A request may charge the payment and still return a timeout, which makes request counts alone insufficient for detecting duplicates.

With idempotency enabled, the service stores a successful payment against a stable key. Reusing that key returns `ok_cached` without creating another charge. Reusing it with a different amount returns `idempotency_conflict`.

This is an in-memory model. It does not implement a payment provider's key retention, persistence, or concurrency rules.

In [ ]:
class SimulatedPaymentService:
    def __init__(self):
        self.success_by_key: Dict[str, int] = {}
        self.charge_count: Counter = Counter()
        self.attempts: Counter = Counter()

    def request(self, payment: Payment, key: str, *, use_idempotency=True):
        self.attempts[payment.payment_id] += 1
        number = self.attempts[payment.payment_id]

        if use_idempotency and key in self.success_by_key:
            if self.success_by_key[key] != payment.amount_cents:
                return "idempotency_conflict"
            return "ok_cached"

        # Repeat the final scripted outcome if attempts exceed the script length.
        outcome = payment.script[min(number - 1, len(payment.script) - 1)]
        if outcome in {"ok", "timeout_after_commit"}:
            self.charge_count[payment.payment_id] += 1
            if use_idempotency:
                self.success_by_key[key] = payment.amount_cents
        return outcome

## 4. Retry policies

The retry client stops after success, a terminal failure, or the attempt limit. Temporary errors can be retried up to four total attempts by default.

Before a retry, the delay bound is:

```text
min(base_delay_ms × 2^(attempt − 1), max_delay_ms)
```

Full jitter samples a delay uniformly between zero and that bound. The default base delay is 100 ms and the cap is 1,000 ms. Each request adds a fixed 45 ms of simulated processing time; the code does not sleep.

The comparison uses three policies: one attempt, retries with a stable idempotency key, and retries without idempotency. The last policy exposes the duplicate-charge risk after an ambiguous timeout.

In [ ]:
@dataclass(frozen=True)
class RetryPolicy:
    name: str
    max_attempts: int = 4
    base_delay_ms: float = 100.0
    max_delay_ms: float = 1000.0
    jitter: bool = True
    use_idempotency: bool = True


def run_policy(payments: List[Payment], policy: RetryPolicy, seed=SEED):
    rng = random.Random(seed)
    service = SimulatedPaymentService()
    rows = []
    attempt_log = []
    for payment in payments:
        key = payment.payment_id  # unchanged across all attempts of this operation
        elapsed_ms = 0.0
        final = None
        for attempt in range(1, policy.max_attempts + 1):
            outcome = service.request(payment, key, use_idempotency=policy.use_idempotency)
            elapsed_ms += 45.0  # synthetic per-request network + processing time
            retryable = outcome in TRANSIENT
            should_retry = retryable and attempt < policy.max_attempts
            delay_ms = 0.0
            if should_retry:
                bound = min(policy.base_delay_ms * (2 ** (attempt - 1)), policy.max_delay_ms)
                delay_ms = rng.uniform(0, bound) if policy.jitter else bound
                elapsed_ms += delay_ms
            attempt_log.append({"payment_id": payment.payment_id, "attempt": attempt,
                                "outcome": outcome, "backoff_ms": delay_ms,
                                "retry_scheduled": should_retry})
            final = outcome
            if not should_retry:
                break
        charges = service.charge_count[payment.payment_id]
        rows.append({"payment_id": payment.payment_id, "amount_usd": payment.amount_cents / 100,
                     "final_outcome": final, "success": final in {"ok", "ok_cached"},
                     "attempts": service.attempts[payment.payment_id],
                     "elapsed_ms": elapsed_ms, "charges": charges,
                     "duplicate_charges": max(charges - 1, 0),
                     "exhausted": final in TRANSIENT and service.attempts[payment.payment_id] == policy.max_attempts})
    return pd.DataFrame(rows), pd.DataFrame(attempt_log)

NO_RETRY = RetryPolicy("No retries", max_attempts=1)
SAFE_RETRY = RetryPolicy("Retries with idempotency")
NO_IDEMPOTENCY = RetryPolicy("Retries without idempotency", use_idempotency=False)

baseline, baseline_log = run_policy(payments, NO_RETRY)
safe, safe_log = run_policy(payments, SAFE_RETRY)
unsafe, unsafe_log = run_policy(payments, NO_IDEMPOTENCY)
display(safe.head(8))

## 5. Policy results

The summary separates what the client knows from what the service charged.

| Metric | Interpretation |
|---|---|
| Success rate | Payments receiving `ok` or `ok_cached` |
| Duplicate charges | Charges beyond the first for each payment |
| Retry exhausted | Payments ending in a temporary error at the attempt limit |
| Total API attempts | Number of simulated requests |
| Mean latency | Average simulated elapsed time, including backoff |
| p95 latency | Simulated elapsed time at the 95th percentile |

A payment can be charged without appearing successful to the client if its response is lost. For the one-attempt baseline, “retry exhausted” means a temporary error remained after that single attempt.

In [ ]:
def summarize(frame, label):
    return {"Policy": label,
            "Success rate (%)": round(100 * frame.success.mean(), 1),
            "Duplicate charges": int(frame.duplicate_charges.sum()),
            "Retry exhausted": int(frame.exhausted.sum()),
            "Total API attempts": int(frame.attempts.sum()),
            "Mean latency (ms)": round(frame.elapsed_ms.mean(), 1),
            "p95 latency (ms)": round(frame.elapsed_ms.quantile(.95), 1)}

results = pd.DataFrame([summarize(baseline, NO_RETRY.name),
                        summarize(safe, SAFE_RETRY.name),
                        summarize(unsafe, NO_IDEMPOTENCY.name)])
display(results)

## 6. Reliability, duplicate charges, and cost

Compare confirmed success and duplicate charges first, then request volume and completion time. A short completion time can mean the client stopped after a failure, so latency should be read alongside success rate.

The delays are modeled rather than measured. This sequential simulation does not show whether jitter reduces congestion among concurrent clients.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results.plot.bar(x="Policy", y="Success rate (%)", ax=axes[0], legend=False, ylim=(0, 105), rot=16)
axes[0].set_title("Confirmed payments")
axes[0].set_ylabel("Successful operations (%)")
results.plot.bar(x="Policy", y="Duplicate charges", ax=axes[1], legend=False, rot=16)
axes[1].set_title("Duplicate charges")
axes[1].set_ylabel("Extra committed charges")
for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results.plot.bar(x="Policy", y="Total API attempts", ax=axes[0], legend=False, rot=16)
axes[0].set_title("Request volume")
axes[0].set_ylabel("Attempts")
results.plot.bar(x="Policy", y="p95 latency (ms)", ax=axes[1], legend=False, rot=16)
axes[1].set_title("p95 simulated completion time")
axes[1].set_ylabel("Milliseconds")
for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 7. Correctness checks

The following six tests check timeout recovery without a duplicate charge, the duplicate risk without idempotency, terminal declines, attempt limits, reproducible jitter within its bounds, and key replay with amount-conflict detection.

Each test uses assertions and stops execution on failure.

In [ ]:
def test_timeout_after_commit_does_not_double_charge():
    p = Payment("timeout_case", 1200, ("timeout_after_commit", "ok", "ok"))
    frame, log = run_policy([p], SAFE_RETRY)
    assert frame.iloc[0].success
    assert frame.iloc[0].charges == 1
    assert frame.iloc[0].duplicate_charges == 0
    assert list(log.outcome) == ["timeout_after_commit", "ok_cached"]


def test_removing_idempotency_can_double_charge():
    p = Payment("unsafe_case", 1200, ("timeout_after_commit", "ok", "ok"))
    frame, _ = run_policy([p], NO_IDEMPOTENCY)
    assert frame.iloc[0].duplicate_charges == 1


def test_decline_is_terminal():
    p = Payment("decline_case", 1200, ("card_declined", "ok"))
    frame, _ = run_policy([p], SAFE_RETRY)
    assert not frame.iloc[0].success
    assert frame.iloc[0].attempts == 1


def test_retry_cap_and_exhaustion():
    p = Payment("always_failing", 1200, ("server_error",) * 5)
    frame, log = run_policy([p], SAFE_RETRY)
    assert frame.iloc[0].attempts == SAFE_RETRY.max_attempts
    assert frame.iloc[0].exhausted
    assert len(log) == SAFE_RETRY.max_attempts
    assert log.iloc[-1].backoff_ms == 0


def test_full_jitter_bounds_and_reproducibility():
    p = Payment("retry_case", 1200, ("rate_limit", "server_error", "ok"))
    one, log1 = run_policy([p], SAFE_RETRY, seed=42)
    two, log2 = run_policy([p], SAFE_RETRY, seed=42)
    assert one.equals(two) and log1.equals(log2)
    assert 0 <= log1.iloc[0].backoff_ms <= 100
    assert 0 <= log1.iloc[1].backoff_ms <= 200


def test_stable_key_replay_and_conflict():
    svc = SimulatedPaymentService()
    a = Payment("a", 1200, ("timeout_after_commit",))
    assert svc.request(a, "key-a") == "timeout_after_commit"
    assert svc.request(a, "key-a") == "ok_cached"
    b = Payment("b", 1500, ("ok",))
    assert svc.request(b, "key-a") == "idempotency_conflict"
    assert sum(svc.charge_count.values()) == 1


tests = [test_timeout_after_commit_does_not_double_charge,
         test_removing_idempotency_can_double_charge,
         test_decline_is_terminal,
         test_retry_cap_and_exhaustion,
         test_full_jitter_bounds_and_reproducibility,
         test_stable_key_replay_and_conflict]
for test in tests:
    test()
    print("PASS", test.__name__)
print(f"\nAll {len(tests)} tests passed.")

## 8. Effect of jitter

Run the same payments with fixed backoff delays instead of randomized delays. Compare completion times and inspect the attempt log for the first ten payments that required another request.

In this model, delays do not change the scripted responses. This experiment measures the difference in waiting time, not the effect of jitter on service capacity.

In [ ]:
EXPERIMENT = RetryPolicy("Retries with fixed backoff", jitter=False)
experiment, experiment_log = run_policy(payments, EXPERIMENT)
display(pd.DataFrame([summarize(safe, SAFE_RETRY.name), summarize(experiment, EXPERIMENT.name)]))

# Inspect the first 10 operations that needed more than one attempt.
interesting_ids = safe.loc[safe.attempts > 1, "payment_id"].head(10)
display(safe_log[safe_log.payment_id.isin(interesting_ids)])

## 9. Interpretation and limitations

Retries can recover temporary failures at the cost of additional requests and waiting time. Idempotency addresses a separate problem: preventing another charge when a previous request succeeded but its response was lost. A declined card remains a terminal result under these policies.

Read the results as behavior of this model, not as a forecast for a production payment system. Requests run sequentially, responses are scripted, processing time is fixed, and idempotency records exist only in memory. The simulation does not cover concurrent requests, key expiry, durable storage, request deadlines, or server-provided `Retry-After` values.

A useful extension would add persistent idempotency records and concurrent requests, then test whether repeated requests can race to create duplicate charges.